# NABILAH DWI INDRIANA
# H061251031 (A)
# Tugas AI Recipe 
-----
# Recipe R-SES-001: Penentuan Hiposenter Gempa dengan Metode Geiger

**Bidang:** Geofisika --- Seismologi\
**Bahasa pemrograman:** Python\
**Lingkungan yang disarankan:** Jupyter Notebook atau Google Colab\
**Status data:** Data simulasi, bukan hasil pengamatan gempa nyata

------------------------------------------------------------------------

## 1. Problem

Hiposenter adalah titik di dalam bumi tempat gempa bumi mulai terjadi.
Tujuan recipe ini adalah memperkirakan koordinat horizontal, kedalaman,
dan waktu asal gempa (*origin time*) berdasarkan waktu tiba gelombang P
dan S yang tercatat di beberapa stasiun seismik.

Recipe ini menggunakan data simulasi agar langkah perhitungan dapat
diperiksa. Data tersebut tidak boleh dianggap sebagai data gempa nyata.

## 2. Minimal Theory

### 2.1 Selisih waktu tiba gelombang P dan S

Untuk satu stasiun, selisih waktu tiba gelombang S dan P adalah

\[ `\Delta `{=tex}t = t_S - t_P \]

Jika kecepatan gelombang P dan S diketahui serta diasumsikan konstan,
perkiraan jarak dapat ditulis sebagai

\[ d = `\frac{\Delta t}{\left(\frac{1}{V_S}-\frac{1}{V_P}\right)}`{=tex}
\]

dengan: - (t_P): waktu tiba gelombang P; - (t_S): waktu tiba gelombang
S; - (V_P): kecepatan gelombang P; - (V_S): kecepatan gelombang S; -
(d): perkiraan jarak dari stasiun ke sumber gempa.

### 2.2 Jarak sumber gempa ke stasiun

Untuk stasiun ke-(i), jarak tiga dimensi dihitung dengan

\[ d_i = `\sqrt{(x-x_i)^2+(y-y_i)^2+h^2}`{=tex} \]

dengan (x,y) sebagai koordinat horizontal sumber gempa, (h) sebagai
kedalaman, dan (x_i,y_i) sebagai koordinat stasiun. Pada contoh ini,
koordinat sumber dan stasiun menggunakan satuan kilometer.

### 2.3 Waktu tiba yang diprediksi

Dengan model kecepatan konstan, waktu tiba yang diprediksi adalah

\[ t\_{P,i}\^{pred}=t_0+`\frac{d_i}{V_P}`{=tex} \]

\[ t\_{S,i}\^{pred}=t_0+`\frac{d_i}{V_S}`{=tex} \]

(t_0) adalah waktu asal gempa.

### 2.4 Residual

Residual merupakan selisih antara waktu tiba yang diamati dan waktu tiba
hasil prediksi:

\[ r_i=t_i^{obs}-t_i^{pred} \]

Residual yang kecil menunjukkan bahwa waktu tiba hasil prediksi lebih
dekat dengan waktu tiba yang digunakan sebagai pengamatan.

### 2.5 Asumsi dan batasan

Recipe ini mengasumsikan kecepatan gelombang seragam di seluruh wilayah.
Model sederhana ini belum memperhitungkan lapisan bumi, pembiasan
gelombang, maupun ketidakpastian pengukuran. Oleh sebab itu, hasilnya
merupakan demonstrasi metode, bukan penentuan hiposenter untuk keperluan
operasional.

## 3. Input dan Constraints

Input yang digunakan: - koordinat (x,y) setiap stasiun seismik dalam
kilometer; - waktu tiba gelombang P dan S dalam detik; - kecepatan
gelombang P dan S dalam kilometer per detik; - tebakan awal koordinat,
kedalaman, dan waktu asal gempa.

Batasan: - data dibuat secara sintetis oleh program; - kecepatan
gelombang dianggap konstan; - perhitungan menggunakan metode iteratif
kuadrat terkecil yang disederhanakan; - pustaka Python yang digunakan
adalah `numpy` dan `matplotlib`.

## 4. Prompt yang Dapat Digunakan Kembali

> Buat program Python untuk memperkirakan hiposenter gempa menggunakan
> pendekatan iteratif metode Geiger. Gunakan koordinat beberapa stasiun,
> waktu tiba gelombang P dan S, serta kecepatan gelombang yang
> diketahui. Program harus memperkirakan koordinat horizontal,
> kedalaman, dan waktu asal gempa; menghitung residual dan RMSE;
> menampilkan grafik lokasi stasiun dan hiposenter; serta memeriksa
> konvergensi. Nyatakan dengan jelas apabila data merupakan data
> simulasi dan jelaskan asumsi serta keterbatasannya.

## 5. Implementasi Python

Salin kode berikut ke satu sel kode di Jupyter Notebook atau Google
Colab, lalu jalankan.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ==========================================================
# 1. DATA STASIUN SEISMIK (koordinat dalam kilometer)
# ==========================================================
stations = np.array([
    [-20.0, -10.0],
    [ 20.0, -15.0],
    [-15.0,  25.0],
    [ 25.0,  20.0],
    [  0.0,  30.0],
    [ 10.0, -25.0]
])

# Kecepatan gelombang (km/s)
Vp = 6.0
Vs = 3.5

# ==========================================================
# 2. MEMBUAT DATA SIMULASI
# Nilai ini hanya digunakan untuk menghasilkan contoh data.
# ==========================================================
true_x = 5.0
true_y = 7.0
true_depth = 12.0
true_t0 = 100.0

true_source = np.array([true_x, true_y, true_depth])

distances_true = np.sqrt(
    (stations[:, 0] - true_source[0])**2
    + (stations[:, 1] - true_source[1])**2
    + true_source[2]**2
)

# Waktu tiba sintetis
tP_obs = true_t0 + distances_true / Vp
tS_obs = true_t0 + distances_true / Vs

# ==========================================================
# 3. TEBakan AWAL MODEL
# Parameter model: x, y, kedalaman, dan waktu asal
# ==========================================================
model = np.array([0.0, 0.0, 8.0, 99.0])

# Gabungkan data P dan S menjadi satu vektor pengamatan
observed = np.concatenate([tP_obs, tS_obs])

max_iterations = 50
tolerance = 1e-6

# ==========================================================
# 4. ITERASI METODE GEIGER SEDERHANA
# ==========================================================
converged = False

for iteration in range(max_iterations):
    x, y, depth, t0 = model

    dx = x - stations[:, 0]
    dy = y - stations[:, 1]
    dz = depth

    distances = np.sqrt(dx**2 + dy**2 + dz**2)

    # Hindari pembagian dengan nol
    distances = np.maximum(distances, 1e-12)

    # Waktu tiba prediksi
    tP_pred = t0 + distances / Vp
    tS_pred = t0 + distances / Vs
    predicted = np.concatenate([tP_pred, tS_pred])

    # Residual = pengamatan - prediksi
    residual = observed - predicted

    # Turunan jarak terhadap x, y, dan kedalaman
    dd_dx = dx / distances
    dd_dy = dy / distances
    dd_dz = depth / distances

    # Matriks Jacobian untuk gelombang P
    Jp = np.column_stack([
        dd_dx / Vp,
        dd_dy / Vp,
        dd_dz / Vp,
        np.ones(len(stations))
    ])

    # Matriks Jacobian untuk gelombang S
    Js = np.column_stack([
        dd_dx / Vs,
        dd_dy / Vs,
        dd_dz / Vs,
        np.ones(len(stations))
    ])

    # Gabungkan persamaan P dan S
    J = np.vstack([Jp, Js])

    # Cari koreksi parameter dengan kuadrat terkecil
    correction, *_ = np.linalg.lstsq(J, residual, rcond=None)

    # Perbarui model
    model = model + correction

    # Berhenti jika perubahan parameter sudah kecil
    if np.linalg.norm(correction) < tolerance:
        converged = True
        break

# ==========================================================
# 5. HITUNG HASIL AKHIR DAN UKURAN KECocokan
# ==========================================================
x_est, y_est, depth_est, t0_est = model

distances_est = np.sqrt(
    (x_est - stations[:, 0])**2
    + (y_est - stations[:, 1])**2
    + depth_est**2
)

tP_pred_final = t0_est + distances_est / Vp
tS_pred_final = t0_est + distances_est / Vs

residual_P = tP_obs - tP_pred_final
residual_S = tS_obs - tS_pred_final
all_residuals = np.concatenate([residual_P, residual_S])
rmse = np.sqrt(np.mean(all_residuals**2))

print("=== HASIL ESTIMASI HIPOSENTER ===")
print(f"Konvergen: {converged}")
print(f"Jumlah iterasi: {iteration + 1}")
print(f"Koordinat X       : {x_est:.4f} km")
print(f"Koordinat Y       : {y_est:.4f} km")
print(f"Kedalaman         : {depth_est:.4f} km")
print(f"Waktu asal gempa  : {t0_est:.4f} s")
print(f"RMSE waktu        : {rmse:.8f} s")

print("\n=== RESIDUAL SETIAP STASIUN ===")
for i in range(len(stations)):
    print(
        f"Stasiun {i+1}: "
        f"Residual P = {residual_P[i]:.8f} s, "
        f"Residual S = {residual_S[i]:.8f} s"
    )

# ==========================================================
# 6. GRAFIK LOKASI HORIZONTAL
# ==========================================================
plt.figure(figsize=(7, 6))
plt.scatter(
    stations[:, 0], stations[:, 1],
    marker="^", s=90, label="Stasiun seismik"
)
plt.scatter(
    x_est, y_est,
    marker="*", s=220, label="Hiposenter hasil estimasi"
)

for i, (sx, sy) in enumerate(stations):
    plt.annotate(f"S{i+1}", (sx, sy), xytext=(5, 5),
                 textcoords="offset points")

plt.xlabel("Koordinat X (km)")
plt.ylabel("Koordinat Y (km)")
plt.title("Stasiun Seismik dan Proyeksi Hiposenter")
plt.axis("equal")
plt.grid(True)
plt.legend()
plt.show()

## 6. Result dan Interpretasi

Program menampilkan koordinat X dan Y, kedalaman, waktu asal gempa,
jumlah iterasi, status konvergensi, RMSE, dan residual gelombang P serta
S untuk setiap stasiun.

Karena waktu tiba dibuat dari sumber simulasi yang sama dengan model
kecepatan konstan, hasil estimasi idealnya mendekati nilai yang
digunakan untuk membuat data, yaitu: - (x=5) km; - (y=7) km; - kedalaman
(=12) km; - waktu asal (t_0=100) detik.

RMSE adalah ukuran rata-rata besar kesalahan waktu dalam bentuk akar
dari rata-rata kuadrat residual. Semakin kecil RMSE, semakin dekat waktu
tiba prediksi dengan data yang digunakan. Residual positif berarti waktu
tiba yang digunakan sebagai pengamatan lebih besar daripada waktu
prediksi; residual negatif berarti waktu pengamatan lebih kecil daripada
waktu prediksi.

Grafik menampilkan posisi stasiun seismik dan proyeksi horizontal
hiposenter. Grafik tersebut tidak menunjukkan kedalaman secara langsung.

## 7. Optional Check

Jalankan kode pemeriksaan berikut setelah kode utama. Pemeriksaan ini
membandingkan hasil estimasi dengan nilai sumber yang digunakan untuk
membuat data simulasi.

In [ ]:
# Pemeriksaan hasil untuk DATA SIMULASI saja
truth = np.array([true_x, true_y, true_depth, true_t0])
estimate = np.array([x_est, y_est, depth_est, t0_est])

print("Nilai simulasi [x, y, kedalaman, t0]:", truth)
print("Nilai estimasi [x, y, kedalaman, t0]:", estimate)
print("Selisih absolut:", np.abs(estimate - truth))

if converged and np.all(np.isfinite(estimate)) and rmse < 0.01:
    print("CHECK: PASS — hasil cocok dengan data simulasi.")
else:
    print("CHECK: PERLU DITINJAU — periksa konvergensi dan residual.")

**Catatan:** hasil `PASS` hanya menunjukkan bahwa program berhasil
merekonstruksi data simulasi dalam batas pemeriksaan yang ditetapkan.
Hal itu bukan bukti bahwa metode sudah tervalidasi untuk data gempa
nyata.

## 8. Keterbatasan dan Pengembangan

Untuk menerapkan metode ini pada gempa nyata, diperlukan waktu tiba P
dan S hasil picking dari seismogram, koordinat stasiun yang sesuai,
serta model kecepatan bawah permukaan yang representatif. Data nyata
dapat mengandung noise, kesalahan picking, dan pengaruh lapisan bumi.
Karena itu, hasil perlu dinilai melalui residual, konvergensi, geometri
sebaran stasiun, dan kesesuaian model kecepatan.

Recipe ini mengikuti struktur umum: problem, teori minimal, input dan
batasan, prompt, kode, hasil, dan pemeriksaan opsional.